In [1]:
import requests
from bs4 import BeautifulSoup

# URL for BBC Business page
business_url = "https://www.bbc.com/business"

# Fetch and parse the page
response = requests.get(business_url)
soup = BeautifulSoup(response.content, "html.parser")

# Try to find all headlines (BBC often uses h3 with class 'gs-c-promo-heading__title')
headlines = soup.find_all("h3", class_="gs-c-promo-heading__title")

# If not found, try anchor tags with class 'gs-c-promo-heading'
if not headlines:
    promo_anchors = soup.select("a.gs-c-promo-heading")
    headlines = [a for a in promo_anchors if a.text.strip()]

# If still not found, fallback to all anchor tags with '/business/' in href and non-empty text
if not headlines:
    headlines = [
        a for a in soup.find_all("a", href=True)
        if "/business/" in a["href"] and a.text.strip()
    ]

if not headlines:
    print("No business headlines found using known selectors.")
else:
    for idx, headline in enumerate(headlines, start=1):
        headline_text = headline.text.strip()
        # Try to get the URL from the parent anchor or from the tag itself
        link = None
        if headline.name == "a" and headline.has_attr("href"):
            link = headline["href"]
        else:
            parent_a = headline.find_parent("a", href=True)
            if parent_a:
                link = parent_a["href"]
        if link and link.startswith("/"):
            link = "https://www.bbc.com" + link
        print(f"{idx}. {headline_text}")
        if link:
            try:
                article_resp = requests.get(link)
                article_soup = BeautifulSoup(article_resp.content, "html.parser")
                # Try to extract all paragraphs in the article body
                article_tag = article_soup.find("article")
                if not article_tag:
                    article_tag = article_soup.find(attrs={"role": "main"})
                if article_tag:
                    paragraphs = article_tag.find_all("p")
                else:
                    paragraphs = article_soup.find_all("p")
                # Combine the text of all paragraphs
                article_text = " ".join([p.get_text(strip=True) for p in paragraphs])
                # Get the first 2-3 sentences for a concise news detail
                import re
                sentences = re.split(r'(?<=[.!?]) +', article_text)
                news_detail = ' '.join(sentences[:3])
                # Try to extract the first image in the article
                img_url = None
                img_tag = article_soup.find('img')
                if img_tag and img_tag.has_attr('src'):
                    img_url = img_tag['src']
                    if img_url.startswith('//'):
                        img_url = 'https:' + img_url
                    elif img_url.startswith('/'):
                        img_url = 'https://www.bbc.com' + img_url
                print(f"   Link: {link}")
                print(f"   News Detail: {news_detail}")
                if img_url:
                    print(f"   Image: {img_url}")
                else:
                    print("   Image: (No image found)")
            except Exception as e:
                print(f"   Link: {link}")
                print(f"   News Detail: (Could not fetch article: {e})")
                print("   Image: (No image found)")
        else:
            print("   Link: (No link found)")
            print("   News Detail: (No article found)")
            print("   Image: (No image found)")

1. Executive Lounge
   News Detail: Nell Diamond, CEO of Hill House Home, shares how her small business skyrocketed to global success with the launch of the Nap Dress in 2019. Cherie Clonan, CEO of The Digital Picnic, explains how free dinners, burnout-preventing days off and even ADHD assessments have boosted employee wellbeing – and her bottom line. Perhaps no app has mastered user loyalty quite like Duolingo, the gamified language-learning platform 34 million people a day can't put down.
2. Technology of Business
   News Detail: Nell Diamond, CEO of Hill House Home, shares how her small business skyrocketed to global success with the launch of the Nap Dress in 2019. Cherie Clonan, CEO of The Digital Picnic, explains how free dinners, burnout-preventing days off and even ADHD assessments have boosted employee wellbeing – and her bottom line. Perhaps no app has mastered user loyalty quite like Duolingo, the gamified language-learning platform 34 million people a day can't put down.
2.